In [55]:
import numpy as np
np.set_printoptions(precision=5)
import matplotlib.pyplot as plt

### Global Variables

In [44]:
sigma_detectorresolution_inner_rphi = 10 * 10**-6   #spatial resolution of the ITS2 inner layers in rphi in meters
sigma_detectorresolution_inner_z = 10 * 10**-6      #spatial resolution of the ITS2 inner layers in z    in meters

sigma_detectorresolution_outer_rphi = 10 * 10**-6   #spatial resolution of the ITS2 outer layers in rphi in meters
sigma_detectorresolution_outer_z = 10 * 10**-6      #spatial resolution of the ITS2 outer layers in z    in meters

layerthickness_inner = 0.0035  # thickness of an inner detector plane in units of radiation length 0.35%
layerthickness_outer = 0.008   # thickness of an outer detector plane in units of radiation length 0.8%

radiation_length_air = 303.9           # radiation length of air in meters

average_layer_radii = np.array([23, 31, 39, 196, 245, 344, 393]) * 10**-3   # average radius of ITS2 layers in meters

magnetic_field_strength = 0.5           # strength of magnetic field in Alice in Tesla

particle_momentum = 1           # momentum of particle in GeV/c
particle_mass = 1               # mass of particle in GeV/c^2         

In [45]:
def get_rms_ScatteringAngle(momentum, mass, material_thickness):             # calculates the variance of the scattering angle of a particle with given momentum and mass and given layer thickness
    energy = np.sqrt(momentum**2 + mass**2)                         # energy of particle in GeV    
    beta = momentum / energy                                        # velocity of particle in units of speed of light c
    f = 0.013 * np.sqrt(material_thickness) * (1 + 0.038*np.log(material_thickness))
    return 1/(beta*momentum) * f 

In [46]:
def get_circ_arc_length(momentum, chord_length):                                # calculates the length of a circular arc with given particle momentum and chord length
    radius_of_curvature = 1/(0.3*magnetic_field_strength) * momentum                              # radius of the curvature of the particle trajectory in meters
    return 2 * radius_of_curvature * np.arcsin(chord_length / (2 * radius_of_curvature))     # length of the circular arc 

In [47]:
def get_air_thickness_straight(radiation_length_air, average_layer_radii, IU_layer):    # helper function to calculate the amount of air in between the ITS layers in units of radiation length
    distances_between_layers = []
    distances_between_layers.append(10**(-90))                                          # to create 7th entry
    for i in range(len(average_layer_radii)-1):
        distances_between_layers.append(average_layer_radii[i+1] - average_layer_radii[i])
    air_thickness = np.array(distances_between_layers) / radiation_length_air
    return air_thickness[IU_layer:]

def get_air_thickness_parabolic(radiation_length_air, average_layer_radii, IU_layer, momentum):    # helper function to calculate the amount of air in between the ITS layers in units of radiation length
    distances_between_layers = []
    distances_between_layers.append(10**(-90))                                          # to create 7th entry
    for i in range(len(average_layer_radii)-1):
        distances_between_layers.append(average_layer_radii[i+1] - average_layer_radii[i])
    distances_between_layers_corr = get_circ_arc_length(momentum, np.array(distances_between_layers))
    air_thickness = np.array(distances_between_layers_corr) / radiation_length_air
    return air_thickness[IU_layer:]

#### Test case
- proton with 1 GeV/c
- 4 hits 
- extrapolation 2.5 cm from the IU layer
- polarangle 90°

In [48]:
N = 4              # Number of layers with hits
r = 0.125            # extrapolation length in meters (is defined away from layers) ;

IU_layer = len(average_layer_radii) - N                                              # starts with layer no.0

def get_layers(inner, outer, IU_layer):                                                    #helper function to get ITS2 layers starting from IU layer
    layer_array = np.array([inner, inner, inner, outer, outer, outer, outer])
    return layer_array[IU_layer:]

sigma_detectorresolution_rphi = get_layers(sigma_detectorresolution_inner_rphi, sigma_detectorresolution_outer_rphi, IU_layer)
sigma_detectorresolution_z = get_layers(sigma_detectorresolution_inner_z, sigma_detectorresolution_outer_z, IU_layer)

print(f"detector resolutions in rhpi : {sigma_detectorresolution_rphi}")
print(f"detector resolutions in z : {sigma_detectorresolution_z}")

layerthickness = get_layers(layerthickness_inner, layerthickness_outer, IU_layer)
sigma_ScatteringAngle_layer = get_rms_ScatteringAngle(particle_momentum, particle_mass, layerthickness)

print(f"layer thicknesses : {layerthickness}")
print(f"Scattering Angles : {sigma_ScatteringAngle_layer}")

air_thickness = get_air_thickness_straight(radiation_length_air, average_layer_radii, IU_layer)
print("thickness of air straight:", air_thickness*100)          # for %
        

air_thickness = get_air_thickness_parabolic(radiation_length_air, average_layer_radii, IU_layer, particle_momentum)
print("thickness of air parabolic:", air_thickness*100)          # for %


sigma_ScatteringAngle_air = get_rms_ScatteringAngle(particle_momentum, particle_mass, air_thickness)
sigma_ScatteringAngle_air[0] = 0
print(f"Scattering Angles in air : {sigma_ScatteringAngle_air}")



sigma_ScatteringAngle_total = sigma_ScatteringAngle_layer + sigma_ScatteringAngle_air
#print(sigma_ScatteringAngle_total)


layer_positions = average_layer_radii[IU_layer:]-average_layer_radii[IU_layer]           #  ITS2 layer positions with r(IU layer) = 0
print(f"layer positions : {layer_positions}")

detector resolutions in rhpi : [1.e-05 1.e-05 1.e-05 1.e-05]
detector resolutions in z : [1.e-05 1.e-05 1.e-05 1.e-05]
layer thicknesses : [0.008 0.008 0.008 0.008]
Scattering Angles : [0.00134 0.00134 0.00134 0.00134]
thickness of air straight: [0.05166 0.01612 0.03258 0.01612]
thickness of air parabolic: [0.05166 0.01612 0.03258 0.01612]
Scattering Angles in air : [0.      0.00016 0.00023 0.00016]
layer positions : [0.    0.049 0.148 0.197]


- detector resolutions in rhpi : [1.e-05 1.e-05 1.e-05 1.e-05]
- detector resolutions in z : [1.e-05 1.e-05 1.e-05 1.e-05]
- layer thicknesses : [0.008 0.008 0.008 0.008]
- Scattering Angles : [0.00134 0.00134 0.00134 0.00134]
- thickness of air straight: [0.05166 0.01612 0.03258 0.01612]
- thickness of air parabolic: [0.05166 0.01612 0.03258 0.01612]
- Scattering Angles in air : [0.      0.00016 0.00023 0.00016]
- layer positions : [0.    0.049 0.148 0.197]

In [56]:
def get_cov_det(detector_resolutions, N):                                 # covarinace matrix due to detector resolution helper function
    cov_det = np.zeros((N, N))
    for n in range(N):
        cov_det[n][n] = detector_resolutions[n]**2
    return cov_det

cov_det = get_cov_det(sigma_detectorresolution_rphi, N)                   # the covariance matrix for detector resolution


def get_cov_MS(sigma_ScatteringAngle, layer_positions, N):                # the covariance matrix due to multiple scattering helper function
    cov_MS = np.zeros((N, N))                                             
    for m in range(N):
        for n in range(N):
            sum = 0      
            for j in range(np.min(np.array([m,n]))):
                sum += sigma_ScatteringAngle[j]**2 * (layer_positions[m] - layer_positions[j]) * (layer_positions[n] - layer_positions[j])
            cov_MS[m][n] = sum
    return cov_MS

cov_MS = get_cov_MS(sigma_ScatteringAngle_total, layer_positions, N)             # the covariance matrix for multiple scattering
cov_MS[0][0] = 10**-99                                                     #  to avoid singular matrix

cov_tot = cov_det + cov_MS                                                 #  total covariance matrix

print("covariance matrix due to detector resolution")
print(cov_det)
print("covariance matrix due to multiple scattering")
print(cov_MS)
#print(cov_tot)

covariance matrix due to detector resolution
[[1.e-10 0.e+00 0.e+00 0.e+00]
 [0.e+00 1.e-10 0.e+00 0.e+00]
 [0.e+00 0.e+00 1.e-10 0.e+00]
 [0.e+00 0.e+00 0.e+00 1.e-10]]
covariance matrix due to multiple scattering
[[1.00000e-99 0.00000e+00 0.00000e+00 0.00000e+00]
 [0.00000e+00 4.32849e-09 1.30738e-08 1.74023e-08]
 [0.00000e+00 1.30738e-08 6.15012e-08 8.54702e-08]
 [0.00000e+00 1.74023e-08 8.54702e-08 1.25103e-07]]


- covariance matrix due to detector resolution

[[1.e-10 0.e+00 0.e+00 0.e+00] $\\$
 [0.e+00 1.e-10 0.e+00 0.e+00] $\\$
 [0.e+00 0.e+00 1.e-10 0.e+00] $\\$
 [0.e+00 0.e+00 0.e+00 1.e-10]]

- covariance matrix due to multiple scattering

[[1.00000e-99 0.00000e+00 0.00000e+00 0.00000e+00] $\\$
 [0.00000e+00 4.32849e-09 1.30738e-08 1.74023e-08] $\\$
 [0.00000e+00 1.30738e-08 6.15012e-08 8.54702e-08] $\\$
 [0.00000e+00 1.74023e-08 8.54702e-08 1.25103e-07]]

In [57]:
def g0(x):               #  trackmodel helper functions 
    return x**0     
def g1(x):
    return x**1
def g2(x):
    return x**2 /2

trackmodel_straight = np.array([g0, g1])     #  straight track model f(x) = a0*g0(x) + a1*g1(x)
trackmodel_parabolic = np.array([g0, g1, g2])    #  parabolic track model f(x) = a0*g0(x) + a1*g1(x) + a2*g2(x)

def get_trackmodel_matrix(layer_positions, trackmodel):   #  trackmodel matrix G helper function
    trackmodel_list = []
    for f in trackmodel:
        trackmodel_list.append(f(layer_positions))
    return np.stack(trackmodel_list).T

trackmodel_matrix_straight = get_trackmodel_matrix(layer_positions, trackmodel_straight)  #  straight trackmodel matrix
print("straight trackmodel matrix")
print(trackmodel_matrix_straight.T)

trackmodel_matrix_parabolic = get_trackmodel_matrix(layer_positions, trackmodel_parabolic)   #  parabolic trackmodel matrix
print("parabolic trackmodel matrix")
print(trackmodel_matrix_parabolic.T)

straight trackmodel matrix
[[1.    1.    1.    1.   ]
 [0.    0.049 0.148 0.197]]
parabolic trackmodel matrix
[[1.      1.      1.      1.     ]
 [0.      0.049   0.148   0.197  ]
 [0.      0.0012  0.01095 0.0194 ]]


- straight trackmodel matrix

[[1.    1.    1.    1.   ] $\\$
 [0.    0.049 0.148 0.197]]

- parabolic trackmodel matrix

[[1.      1.      1.      1.     ] $\\$
 [0.      0.049   0.148   0.197  ] $\\$
 [0.      0.0012  0.01095 0.0194 ]]

In [51]:
def get_cov_para(cov, trackmodel_matrix):                                                 # Covariance matrix of parameters helper function
    return np.linalg.inv(trackmodel_matrix.T @ np.linalg.inv(cov) @ trackmodel_matrix)

# straight track model
cov_para_det_straight = get_cov_para(cov_det, trackmodel_matrix_straight)   #  C_a contribution from detector resolution
cov_para_MS_straight = get_cov_para(cov_MS, trackmodel_matrix_straight)     #  C_a contribution from MS

# parabolic track model
cov_para_det_parabolic = get_cov_para(cov_det, trackmodel_matrix_parabolic)   #  C_a contribution from detector resolution
cov_para_MS_parabolic = get_cov_para(cov_MS, trackmodel_matrix_parabolic)        #  C_a contribution from MS

In [59]:
def apply(trackmodel, extrapolation_radius):                # helper function to apply array of functions on one argument
    _list = []
    for f in trackmodel:
        _list.append(f(extrapolation_radius))
    return np.array(_list)

def get_pos_reso(cov_para, trackmodel, extrapolation_radius):                                                             # Position Resolution helper function
    return np.sqrt(apply(trackmodel, -extrapolation_radius).T @ cov_para @ apply(trackmodel, -extrapolation_radius))

def get_momentum_reso(cov_para, momentum, magnetic_field_strength):                                                      # Momentum Resolution helper function
    return momentum / (0.3*magnetic_field_strength) * np.sqrt(cov_para[2][2])


def get_pos_reso_MS_extrapolation_straight(momentum, mass, extrapolation_radius, radiation_length_air):                         # Position resolution contribution from MS in air during extrapolation (straight extrapolation simulated)
    air_thickness_extrapolation = extrapolation_radius / radiation_length_air
    sigma_ScatteringAngle_extrapolation = get_rms_ScatteringAngle(momentum, mass, air_thickness_extrapolation)
    posreso_MS_extrapolation = sigma_ScatteringAngle_extrapolation * extrapolation_radius
    return posreso_MS_extrapolation

def get_pos_reso_MS_extrapolation_parabolic(momentum, mass, extrapolation_radius, radiation_length_air):                # Position resolution contribution from MS in air during extrapolation (parabolic extrapolation simulated)
    circ_length = get_circ_arc_length(momentum, extrapolation_radius)
    print(circ_length)
    air_thickness_extrapolation = circ_length / radiation_length_air
    sigma_ScatteringAngle_extrapolation = get_rms_ScatteringAngle(momentum, mass, air_thickness_extrapolation)
    posreso_MS_extrapolation = sigma_ScatteringAngle_extrapolation * circ_length
    return posreso_MS_extrapolation

posreso_MS_extrapolation_straight = get_pos_reso_MS_extrapolation_straight(particle_momentum, particle_mass, r, radiation_length_air)
print("position resolution from straigt extraploation", posreso_MS_extrapolation_straight)
posreso_MS_extrapolation_parabolic = get_pos_reso_MS_extrapolation_parabolic(particle_momentum, particle_mass, r, radiation_length_air)
print("position resolution from parabolic extraploation",posreso_MS_extrapolation_parabolic)

position resolution from straigt extraploation 3.2800039972824913e-05
0.12500183112711094
position resolution from parabolic extraploation 3.280078665313987e-05


In [53]:
# straight track model
print("straight model:")
posreso_det_straight = get_pos_reso(cov_para_det_straight, trackmodel_straight, r)
posreso_MS_straight = np.sqrt(get_pos_reso(cov_para_MS_straight, trackmodel_straight, r)**2 + posreso_MS_extrapolation_straight**2)
print("contribution from detector resolution", posreso_det_straight)
print("contribution from multiple scattering", posreso_MS_straight)


print()
# parabolic track model
print("parabolic model:")
posreso_det_parabolic = get_pos_reso(cov_para_det_parabolic, trackmodel_parabolic, r)
posreso_MS_parabolic = np.sqrt(get_pos_reso(cov_para_MS_parabolic, trackmodel_parabolic, r)**2 + posreso_MS_extrapolation_parabolic**2)    
print("Position Resolution:")
print("contribution from detector resolution", posreso_det_parabolic)
print("contribution from multiple scattering", posreso_MS_parabolic)

momentumreso_det_parabolic = get_momentum_reso(cov_para_det_parabolic, particle_momentum, magnetic_field_strength)
momentumreso_MS_parabolic = get_momentum_reso(cov_para_MS_parabolic, particle_momentum, magnetic_field_strength)

print("Momentum Resolution:")
print("contribution from detector resolution", momentumreso_det_parabolic)
print("contribution from multiple scattering" , momentumreso_MS_parabolic)

straight model:
contribution from detector resolution 1.5182968294413673e-05
contribution from multiple scattering 0.0001710099600946046

parabolic model:
Position Resolution:
contribution from detector resolution 6.23779297563619e-05
contribution from multiple scattering 0.00023382703876488048
Momentum Resolution:
contribution from detector resolution 0.018385732671447004
contribution from multiple scattering 0.0977592864930634


- **straight model:**
- - contribution from detector resolution 1.5182968294413673e-05
- - contribution from multiple scattering 0.0001710099600946046

- **parabolic model:**
- Position Resolution:
- - contribution from detector resolution 6.23779297563619e-05
- - contribution from multiple scattering 0.00023382703876488048
- Momentum Resolution:
- - contribution from detector resolution 0.018385732671447004
- - contribution from multiple scattering 0.0977592864930634

In [54]:
print("added quadratically:")

posreso_tot_straight_test = np.sqrt(posreso_det_straight**2 + posreso_MS_straight**2)
print("positon resolution in z in m", posreso_tot_straight_test)

posreso_tot_parabolic_test = np.sqrt(posreso_det_parabolic**2 + posreso_MS_parabolic**2)
print("positon resolution in rphi in m",posreso_tot_parabolic_test)

momentumreso_tot_test = np.sqrt(momentumreso_det_parabolic**2 + momentumreso_MS_parabolic**2)
print("Momentum Resolution:", momentumreso_tot_test)



print()
print("from total cov:")

cov_para_tot_straight = get_cov_para(cov_tot, trackmodel_matrix_straight)  # straight track model
cov_para_tot_parabolic = get_cov_para(cov_tot, trackmodel_matrix_parabolic)  # parabolic track model

posreso_tot_straight = get_pos_reso(cov_para_tot_straight, trackmodel_straight, r)   # straight track model
posreso_tot_parabolic = get_pos_reso(cov_para_tot_parabolic, trackmodel_parabolic, r)    # parabolic track model

print("positon resolution in z", posreso_tot_straight)
print("positon resolution in rphi", posreso_tot_parabolic)

momentumreso_tot = get_momentum_reso(cov_para_tot_parabolic, particle_momentum, magnetic_field_strength)
print("Momentum Resolution:", momentumreso_tot)

added quadratically:
positon resolution in z in m 0.0001716826402924519
positon resolution in rphi in m 0.0002420043185115558
Momentum Resolution: 0.09947317860357514

from total cov:
positon resolution in z 0.00017315887590331304
positon resolution in rphi 0.00024145495639537291
Momentum Resolution: 0.09948230748591853


Reference from Thesis:

Proton with ~ 1GeV/c:
- r = 0.025:  &nbsp; &nbsp; &nbsp;   0.005 cm   &nbsp; &nbsp; &nbsp;  5 * 10^-5 m 
- r = 0.125:  &nbsp; &nbsp; &nbsp;   0.020 cm   &nbsp; &nbsp; &nbsp;  2 * 10^-4 m